# TFT Scouting — Notebook 3: Meta Analysis

**Objetivo:** A partir del NDJSON crudo, extraer:
1. Win rates de composiciones (top-4 rate por trait dominante)
2. Items más frecuentes por carry
3. Augments por stage y su correlación con placement
4. Señales de variantes: misma comp core → distintos units/items → distintos placements

Este análisis es el insumo para el prompt de generación de guías con IA.

In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

# ── Cargar datos ──────────────────────────────────────────────────────────────
matches = []
with open('matches_raw.ndjson', 'r') as f:
    for line in f:
        matches.append(json.loads(line))

print(f'Partidas cargadas: {len(matches)}')

Partidas cargadas: 2745


In [2]:
# ── Aplanar participantes ─────────────────────────────────────────────────────

rows = []
for match in matches:
    match_id = match['metadata']['match_id']
    game_datetime = match['info'].get('game_datetime', 0)
    for p in match['info']['participants']:
        rows.append({
            'match_id':       match_id,
            'game_datetime':  game_datetime,
            'puuid':          p.get('puuid'),
            'placement':      p.get('placement'),
            'level':          p.get('level'),
            'last_round':     p.get('last_round'),
            'total_damage_to_players': p.get('total_damage_to_players', 0),
            'players_eliminated': p.get('players_eliminated', 0),
            'augments':       p.get('augments', []),
            'traits':         p.get('traits', []),
            'units':          p.get('units', []),
        })

df = pd.DataFrame(rows)
df['top4'] = df['placement'] <= 4
df['win']  = df['placement'] == 1

print(f'Participantes totales: {len(df)}')
df.head(2)

Participantes totales: 21883


,match_id,game_datetime,puuid,placement,level,last_round,total_damage_to_players,players_eliminated,augments,traits,units,top4,win
0,EUW1_7761291413,1772515096867,UyM8O0EJnOuxoE9IR3JUkQWjK076_cW_Ku9OLOCHHK-kek...,3,9,35,110,1,[],"[{'name': 'TFT16_Brawler', 'num_units': 2, 'st...","[{'character_id': 'TFT16_Shen', 'itemNames': [...",True,False
1,EUW1_7761291413,1772515096867,cIS7pCMhM9ugJZcxVqpKmFxJH3qNNX4JKefGin6ZKa91_N...,8,8,23,11,0,[],"[{'name': 'TFT16_Brawler', 'num_units': 2, 'st...","[{'character_id': 'TFT16_KogMaw', 'itemNames':...",False,False


In [3]:
# ── 1. Traits activos (tier >= 1) por participante ────────────────────────────
# Un 'trait dominante' es el trait con mayor num_units que define la comp

def get_active_traits(traits):
    """Devuelve lista de (trait_id, tier_current, num_units) activos."""
    return [
        (t['name'], t.get('tier_current', 0), t.get('num_units', 0))
        for t in traits
        if t.get('tier_current', 0) > 0
    ]

def get_dominant_trait(traits):
    """El trait con más units activos — proxy del 'nombre' de la comp."""
    active = get_active_traits(traits)
    if not active:
        return 'Unknown'
    return max(active, key=lambda x: x[2])[0]

def get_trait_signature(traits, top_n=3):
    """Top N traits por units activos — identifica la comp más precisamente."""
    active = sorted(get_active_traits(traits), key=lambda x: x[2], reverse=True)
    return tuple(t[0] for t in active[:top_n])

df['dominant_trait']  = df['traits'].apply(get_dominant_trait)
df['trait_signature'] = df['traits'].apply(get_trait_signature)

df[['dominant_trait', 'trait_signature', 'placement', 'top4']].head(10)

,dominant_trait,trait_signature,placement,top4
0,TFT16_Ionia,"(TFT16_Ionia, TFT16_Brawler, TFT16_Juggernaut)",3,True
1,TFT16_Juggernaut,"(TFT16_Juggernaut, TFT16_Void, TFT16_Brawler)",8,False
2,TFT16_Zaun,"(TFT16_Zaun, TFT16_Brawler, TFT16_Invoker)",4,True
3,TFT16_Piltover,"(TFT16_Piltover, TFT16_Defender, TFT16_Gunslin...",2,True
4,TFT16_Defender,"(TFT16_Defender, TFT16_Magus, TFT16_Piltover)",7,False
5,TFT16_Demacia,"(TFT16_Demacia, TFT16_Juggernaut, TFT16_Defender)",5,False
6,TFT16_Ionia,"(TFT16_Ionia, TFT16_Brawler, TFT16_Defender)",1,True
7,TFT16_Defender,"(TFT16_Defender, TFT16_Juggernaut, TFT16_Magus)",6,False
8,TFT16_Ionia,"(TFT16_Ionia, TFT16_Brawler, TFT16_Defender)",6,False
9,TFT16_Freljord,"(TFT16_Freljord, TFT16_Zaun, TFT16_Defender)",4,True


In [4]:
# ── 2. Top-4 rate por trait dominante ─────────────────────────────────────────

trait_stats = (
    df.groupby('dominant_trait')
    .agg(
        games       = ('placement', 'count'),
        top4_rate   = ('top4', 'mean'),
        avg_placement = ('placement', 'mean'),
        win_rate    = ('win', 'mean')
    )
    .query('games >= 20')  # Mínimo de muestra
    .sort_values('top4_rate', ascending=False)
    .round(3)
)

print('=== Top traits por top-4 rate ===')
print(trait_stats.head(15).to_string())

=== Top traits por top-4 rate ===
                  games  top4_rate  avg_placement  win_rate
dominant_trait                                             
Unknown             254      0.768          2.760     0.563
TFT16_Shurima       137      0.759          2.964     0.350
TFT16_Warden        737      0.586          4.213     0.098
TFT16_Slayer        275      0.585          4.178     0.069
TFT16_Gunslinger     33      0.576          3.909     0.242
TFT16_Sorcerer     1068      0.567          4.135     0.172
TFT16_Juggernaut    778      0.553          4.350     0.104
TFT4_Elderwood       22      0.545          4.318     0.182
TFT16_Void          691      0.544          4.207     0.158
TFT16_Brawler      1123      0.527          4.408     0.109
TFT16_Vanquisher    112      0.527          4.250     0.116
TFT16_Freljord     1338      0.519          4.406     0.121
TFT16_Rapidfire      31      0.516          4.323     0.161
TFT16_Targon        442      0.514          4.407     0.113
TFT16_

In [6]:
# ── DIAGNÓSTICO: ver estructura real de units ─────────────────────────────────

with open('matches_raw.ndjson', 'r') as f:
    sample = json.loads(f.readline())

p = sample['info']['participants'][0]
print("Keys del participante:", list(p.keys()))
print("\nPrimer unit completo:")
print(json.dumps(p['units'][0], indent=2))
print("\nTotal units en este participante:", len(p['units']))

Keys del participante: ['companion', 'gold_left', 'last_round', 'level', 'missions', 'placement', 'players_eliminated', 'puuid', 'riotIdGameName', 'riotIdTagline', 'time_eliminated', 'total_damage_to_players', 'traits', 'units', 'win']

Primer unit completo:
{
  "character_id": "TFT16_Shen",
  "itemNames": [],
  "name": "",
  "rarity": 0,
  "tier": 2
}

Total units en este participante: 9


In [7]:
# ¿Dónde están los augments?
p = sample['info']['participants'][0]
print("¿augments en participant?", 'augments' in p)

# Buscar en info general
print("Keys de info:", list(sample['info'].keys()))

# Ver si están en otro nivel
if 'augments' in sample['info']:
    print("augments en info:", sample['info']['augments'])

¿augments en participant? False
Keys de info: ['endOfGameResult', 'gameCreation', 'gameId', 'game_datetime', 'game_length', 'game_version', 'mapId', 'participants', 'queueId', 'queue_id', 'tft_game_type', 'tft_set_core_name', 'tft_set_number']


In [8]:
# ── Buscar augments en toda la estructura ─────────────────────────────────────

import json

with open('matches_raw.ndjson', 'r') as f:
    sample = json.loads(f.readline())

p = sample['info']['participants'][0]

# 1. Buscar en units
print("=== Keys de TODOS los units del participante 0 ===")
for i, u in enumerate(p['units']):
    print(f"Unit {i} ({u['character_id']}):", list(u.keys()))

# 2. Buscar recursivamente "augment" en cualquier campo
def find_key(obj, search, path=""):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if search.lower() in k.lower():
                print(f"  ENCONTRADO → {path}.{k}: {v}")
            find_key(v, search, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            find_key(v, search, f"{path}[{i}]")

print("\n=== Búsqueda recursiva de 'augment' en participant 0 ===")
find_key(p, 'augment')

print("\n=== Búsqueda recursiva de 'augment' en info (nivel raíz) ===")
find_key(sample['info'], 'augment')

=== Keys de TODOS los units del participante 0 ===
Unit 0 (TFT16_Shen): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 1 (TFT16_Ahri): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 2 (TFT16_Wukong): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 3 (TFT16_Yunara): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 4 (TFT16_Swain): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 5 (TFT16_Kindred): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 6 (TFT16_Lucian): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 7 (TFT16_Shyvana): ['character_id', 'itemNames', 'name', 'rarity', 'tier']
Unit 8 (TFT16_Sett): ['character_id', 'itemNames', 'name', 'rarity', 'tier']

=== Búsqueda recursiva de 'augment' en participant 0 ===

=== Búsqueda recursiva de 'augment' en info (nivel raíz) ===


In [9]:
# 1. Ver el campo missions — a veces meten datos ahí
print("missions:", json.dumps(p.get('missions', {}), indent=2))

# 2. Revisar un participante diferente — el 0 puede ser edge case
p2 = sample['info']['participants'][3]
print("\nParticipant 3 keys:", list(p2.keys()))
find_key(p2, 'augment')

# 3. Revisar en otra partida distinta
with open('matches_raw.ndjson', 'r') as f:
    for i, line in enumerate(f):
        if i == 50:  # Partida 50
            sample2 = json.loads(line)
            break

p3 = sample2['info']['participants'][0]
print("\nPartida 50, participant 0:")
find_key(p3, 'augment')
print("Keys:", list(p3.keys()))

missions: {
  "PlayerScore2": 216
}

Participant 3 keys: ['companion', 'gold_left', 'last_round', 'level', 'missions', 'placement', 'players_eliminated', 'puuid', 'riotIdGameName', 'riotIdTagline', 'time_eliminated', 'total_damage_to_players', 'traits', 'units', 'win']

Partida 50, participant 0:
Keys: ['companion', 'gold_left', 'last_round', 'level', 'missions', 'placement', 'players_eliminated', 'puuid', 'riotIdGameName', 'riotIdTagline', 'time_eliminated', 'total_damage_to_players', 'traits', 'units', 'win']


In [10]:
# ¿Hay más datos en missions?
with open('matches_raw.ndjson', 'r') as f:
    for i, line in enumerate(f):
        match = json.loads(line)
        for p in match['info']['participants']:
            missions = p.get('missions', {})
            if len(missions) > 1:
                print(f"Partida {i}, missions con {len(missions)} campos:", missions)
                break
        if i > 100:
            break

In [ ]:
# ── 3. Items más frecuentes en carries ───────────────────────────────────────

def get_carries(units):
    """Units con 2+ items — usa itemNames según API actual."""
    return [
        (u['character_id'], u.get('itemNames', []), u.get('rarity', 0), u.get('tier', 1))
        for u in units
        if len(u.get('itemNames', [])) >= 2
    ]

carry_item_data = []
for _, row in df[df['top4']].iterrows():
    for champ, items, rarity, tier in get_carries(row['units']):
        for item in items:
            carry_item_data.append({
                'champion': champ,
                'item':     item,
                'tier':     tier,
                'placement': row['placement']
            })

print(f'Registros carry-item: {len(carry_item_data)}')

if len(carry_item_data) == 0:
    print("Sin datos — revisar itemNames en units")
else:
    df_items = pd.DataFrame(carry_item_data)
    carry_items = (
        df_items.groupby(['champion', 'item'])
        .agg(frequency=('item', 'count'))
        .reset_index()
        .sort_values(['champion', 'frequency'], ascending=[True, False])
        .groupby('champion')
        .head(3)
    )
    top_carries = df_items.groupby('champion').size().sort_values(ascending=False).head(10).index
    print('=== Items más frecuentes por carry (en top-4) ===')
    print(carry_items[carry_items['champion'].isin(top_carries)].to_string())

KeyError: 'champion'

In [ ]:
# ── 4. Augments por stage y correlación con placement ─────────────────────────
# Los augments en la API vienen como lista ordenada: [stage2, stage3, stage4]
# Índice 0 → 2-1, Índice 1 → 3-2, Índice 2 → 4-2

augment_rows = []
for _, row in df.iterrows():
    for i, aug in enumerate(row['augments']):
        augment_rows.append({
            'augment':    aug,
            'stage':      f'2-{i+1}' if i == 0 else f'{i+2}-2',
            'stage_idx':  i,
            'placement':  row['placement'],
            'top4':       row['top4']
        })

df_aug = pd.DataFrame(augment_rows)

aug_stats = (
    df_aug.groupby(['stage_idx', 'augment'])
    .agg(
        games         = ('top4', 'count'),
        top4_rate     = ('top4', 'mean'),
        avg_placement = ('placement', 'mean')
    )
    .query('games >= 15')
    .sort_values(['stage_idx', 'top4_rate'], ascending=[True, False])
    .round(3)
)

print('=== Top augments por stage (top-4 rate) ===')
for stage_idx in [0, 1, 2]:
    stage_label = ['2-1', '3-2', '4-2'][stage_idx]
    print(f'\n--- Stage {stage_label} ---')
    print(aug_stats.xs(stage_idx, level='stage_idx').head(8).to_string())

In [ ]:
# ── 5. Variantes de comp: mismo core → distintos placements ───────────────────
# Esta sección es clave para generar las guías de variantes

# Definimos comp core como los 3 units con más copias (tier >= 2)
def get_comp_core(units, top_n=3):
    upgraded = sorted(
        [u for u in units if u.get('tier', 1) >= 2],
        key=lambda x: (x.get('tier', 1), x.get('rarity', 0)),
        reverse=True
    )
    return frozenset(u['character_id'] for u in upgraded[:top_n])

df['comp_core'] = df['units'].apply(get_comp_core)
df['comp_core_str'] = df['comp_core'].apply(lambda x: '|'.join(sorted(x)))

# Comps core con suficiente muestra
core_counts = df['comp_core_str'].value_counts()
valid_cores = core_counts[core_counts >= 20].index

print(f'Comp cores con 20+ partidas: {len(valid_cores)}')
print('\nTop 10 comp cores más jugados:')
print(core_counts.head(10).to_string())

In [ ]:
# ── 6. Generar resumen por comp para el prompt de IA ─────────────────────────

comp_summaries = {}

for core_str in valid_cores[:10]:  # Primero las 10 más jugadas
    subset = df[df['comp_core_str'] == core_str]
    
    # Stats generales
    top4_rate = subset['top4'].mean()
    avg_placement = subset['placement'].mean()
    n_games = len(subset)
    
    # Augments más frecuentes en top-4
    top4_subset = subset[subset['top4']]
    aug_counter = Counter(
        aug for augs in top4_subset['augments'] for aug in augs
    )
    top_augments = aug_counter.most_common(6)
    
    # Traits activos más frecuentes en top-4
    trait_counter = Counter()
    for traits in top4_subset['traits']:
        for t in traits:
            if t.get('tier_current', 0) > 0:
                trait_counter[t['name']] += 1
    top_traits = trait_counter.most_common(5)
    
    comp_summaries[core_str] = {
        'core_units':    core_str.split('|'),
        'n_games':       n_games,
        'top4_rate':     round(top4_rate, 3),
        'avg_placement': round(avg_placement, 2),
        'top_augments':  top_augments,
        'top_traits':    top_traits,
    }

# Mostrar ejemplo
best_comp = max(comp_summaries, key=lambda x: comp_summaries[x]['top4_rate'])
print(f'Mejor comp core: {best_comp}')
print(json.dumps(comp_summaries[best_comp], indent=2, default=str))

In [ ]:
# ── 7. Guardar summaries para el prompt de generación ─────────────────────────

with open('comp_summaries.json', 'w') as f:
    json.dump(comp_summaries, f, indent=2, default=str)

# También exportar trait stats y aug stats para BigQuery
trait_stats.to_csv('trait_stats.csv')
aug_stats.to_csv('aug_stats.csv')
carry_items.to_csv('carry_items.csv', index=False)

print('Archivos guardados: comp_summaries.json, trait_stats.csv, aug_stats.csv, carry_items.csv')

## Siguiente paso: Notebook 4 (Guide Generator)

Con `comp_summaries.json` ya tienes el insumo para hacer el prompt a Claude/Gemini:

```python
prompt = f"""
Eres un experto en TFT Set 16 (Lore & Legends). 
Basándote en los siguientes datos reales de partidas de jugadores Challenger/Grandmaster de EUW,
genera una guía completa para la composición con core: {comp['core_units']}

DATOS:
- Top-4 rate: {comp['top4_rate']*100:.1f}%
- Placement promedio: {comp['avg_placement']}
- Partidas analizadas: {comp['n_games']}
- Augments más elegidos en top-4: {comp['top_augments']}
- Traits activos más frecuentes: {comp['top_traits']}

Genera:
1. Descripción de la comp (2-3 líneas)
2. Gameplan por stage (2-1, 3-2, 4-2)
3. Items recomendados para cada carry
4. 2-3 variantes según lo que encuentres en partida
5. Señales de cuándo pivotar a otra comp

Responde en español. Formato JSON.
"""
```